In [ ]:
######## LIBRARIES
import pydicom 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog, simpledialog
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.weightstats import DescrStatsW

In [ ]:
######## SEARCH WINDOW
def select_file(
        title="Select file",
        filetypes=(("All files", "*.*"),)
        ):
    root = tk.Tk()
    root.withdraw()
    file = filedialog.askopenfilename(
        title=title,
        initialdir="C:/",
        filetypes=filetypes
    )
    root.destroy()
    return file

In [ ]:
######## LOG FILE (CSV) SELECTION WINDOW
def select_csvs(TvsA):
    root = tk.Tk()
    root.withdraw()
    
    # Number of log files (CSV)
    num_csv = simpledialog.askinteger(
        f"Log files {TvsA}",
        "Number of log files in this session",
        minvalue = 1,  # Min. number of CSV
        maxvalue = 5   #Max. number of CSV
    )
    
    if num_csv is None:
        print("ERROR.")
        root.destroy()
        return []
    files_csv = []
    for i in range(num_csv):
        root_csv = select_file(
            title=f"Select log file (csv) number {i+1} from this session",
            filetypes=(("CSV files", "*.csv"),)
        )
        if root_csv:
            files_csv.append(root_csv)
            
    root.destroy()
    return files_csv

In [ ]:
######## CONCATENATE LOG FILES (CSV)
def concat_csvs(root_csv):
    if not root_csv:
        print("No log file was selected")
        return None

    # Read all DataFrames in a list
    dataframes = [pd.read_csv(root, skiprows=9) for root in root_csv]

    # If there is only one log file, open it directly
    if len(dataframes) == 1:
        print("Only one log file was selected")
        return dataframes[0]

    # Column name
    col_time = "Time (s)"
    col_cp = "Control point/Actual Value (None)"

    # Continuity between archives
    for i in range(1, len(dataframes)):
        last_time_record = dataframes[i-1][col_time].iloc[-1]
        last_cp_record = dataframes[i-1][col_cp].iloc[-1]

        dataframes[i][col_time] += last_time_record
        dataframes[i][col_cp] += last_cp_record

    # Concatenate all DataFrames
    df_concat = pd.concat(dataframes, ignore_index=True)
    return df_concat

In [ ]:
######## DICOM DATA (DCM)
def dcm_data(root_dcm): 
    dcm = pydicom.dcmread(
        root_dcm, 
        force=True
        )
    rows = []
    DCM_Dose = 0
    DCM_CP = 1
    # Info of treatment & pacient
    P_Name = dcm.PatientName
    P_ID = dcm.PatientID
    Intent = dcm.PlanIntent
    
    # Treatment target
    if hasattr(dcm, 'DoseReferenceSequence'):
        Target = dcm.DoseReferenceSequence[0].DoseReferenceDescription

    # Info per beam
    for beam in dcm.BeamSequence:
        BN = beam.BeamNumber

        # Total dose per beam
        DCM_BD = None
        for fg in dcm.FractionGroupSequence:
            for r_beam in fg.ReferencedBeamSequence:
                if r_beam.ReferencedBeamNumber == BN:
                    DCM_BD = r_beam.BeamMeterset
                    break
            if DCM_BD is not None:
                break

        # Total dose
        DCM_Dose += DCM_BD
        
        # Gantry angle
        Gantry = beam.ControlPointSequence[0].GantryAngle

        # Info per control point
        for i, cp in enumerate(beam.ControlPointSequence):
            row = {
                "Beam Number": BN,
                "Control Point": DCM_CP,
                "Beam Dose": DCM_BD,
                "Total Dose [MU]": DCM_Dose,
                "Gantry Angle": Gantry,
            }

            # Dose per cp
            MW = cp.CumulativeMetersetWeight
            if i == 0:
                MU_cp = 0
            else:
                MW_prev = beam.ControlPointSequence[i-1].CumulativeMetersetWeight
                MU_cp = (MW - MW_prev) * DCM_BD
            row["Dose per CP"] = MU_cp

            # Info Jaws & MLC
            if hasattr(cp, "BeamLimitingDevicePositionSequence"):
                for bld in cp.BeamLimitingDevicePositionSequence:
                    device_type = bld.RTBeamLimitingDeviceType
                    pos = np.array(bld.LeafJawPositions)

                    # Jaws
                    if device_type in ("ASYMX", "ASYMY") and pos.size == 2:
                        row["JAW X2"] = float(pos[0])
                        row["JAW X1"] = float(pos[1])

                    # MLCs
                    if device_type in ("MLCX", "MLCY") and pos.size == 160:
                        DCM_MLC_L = pos[:80]
                        DCM_MLC_R = pos[80:]
                        for j in range(80):
                            row[f"MLC Y1: {j+1}"] = DCM_MLC_L[j]
                        for j in range(80):
                            row[f"MLC Y2: {j+1}"] = DCM_MLC_R[j]

            rows.append(row)
            DCM_CP += 1

    # DataFrame with DCM info
    df_DCM = pd.DataFrame(rows).iloc[:-1]
    
    # Info per column
    DCM_CP_DF = df_DCM["Control Point"].tolist()
    DCM_TD = df_DCM["Total Dose [MU]"]
    DCM_NH = df_DCM["Beam Number"]
    DCM_BD = df_DCM["Beam Dose"]
    DCM_DPC = df_DCM["Dose per CP"]
    DCM_GANTRY = df_DCM["Gantry Angle"]
    DCM_JAW_X1 = df_DCM["JAW X1"]
    DCM_JAW_X2 = df_DCM["JAW X2"]


    # Left MLC
    DCM_Y1_C = df_DCM.loc[:, "MLC Y1: 1":"MLC Y1: 80"].iloc[:, ::-1]
    # Rigth MLC
    DCM_Y2_C = df_DCM.loc[:, "MLC Y2: 1":"MLC Y2: 80"].iloc[:, ::-1]

    # Area per CP
    DCM_ACP = ((DCM_Y2_C.values - DCM_Y1_C.values)*5).sum(axis=1).tolist()
    df_DCM["Area per CP"] = DCM_ACP
    
    # Dictionary with all data
    data = {
        "Control Point": DCM_CP_DF,
        "Beam Number": DCM_NH,
        "Total Dose": DCM_TD,
        "Dose per Beam": DCM_BD,
        "Dose per CP": DCM_DPC,
        "Area per CP": DCM_ACP,
        "Gantry Angle": DCM_GANTRY,
        "Jaw X1": DCM_JAW_X1,
        "Jaw X2": DCM_JAW_X2,
        "MLC Y1": DCM_Y1_C,
        "MLC Y2": DCM_Y2_C
    }

    patient = {
        "Patient Name": P_Name,
        "Patient ID": P_ID,
        "Target": Target,
        "Intent": Intent
    }

    return df_DCM, data, patient

In [ ]:
######## LOG FILE (CSV) DATA
def csv_data(df_CSV):
    # Time
    Time_c = "Time (s)"
    Time = df_CSV[Time_c]

    # Control Points
    CSV_CP_C = "Control point/Actual Value (None)"
    CSV_CP = df_CSV[CSV_CP_C].unique().tolist()

    # Linac state
    LS_C = "Linac State/Actual Value (None)"
    LS = df_CSV[LS_C]

    # Filter by linac state
    F_RO = df_CSV[LS == "Radiation On"] 
    F_MO = df_CSV[LS == "Move Only"]     

    # Dose
    DR_C = "Actual Dose Rate/Actual Value (Mu/min)"
    DR_d = df_CSV[DR_C]
    F_DR = df_CSV[DR_d != 0]  

    AD_C = "Step Dose/Actual Value (Mu)"
    CSV_D_CP = F_RO.groupby(CSV_CP_C)[AD_C].max()
    CSV_TD = CSV_D_CP.sum()

    # Gantry
    CSV_GTY_C = "Step Gantry/Scaled Actual (deg)"
    GANTRY = F_RO.groupby(CSV_CP_C)[CSV_GTY_C].mean()
    CSV_GANTRY = GANTRY.where(GANTRY > 0, GANTRY + 360)

    # Jaws
    JAW_X1 = "X1 Diaphragm/Scaled Actual (mm)"
    JAW_X2 = "X2 Diaphragm/Scaled Actual (mm)"
    CSV_JAWS_1 = F_RO.groupby(CSV_CP_C)[JAW_X1].mean()
    CSV_JAWS_2 = (-1) * F_RO.groupby(CSV_CP_C)[JAW_X2].mean()

    # Left MLC
    I_CY1 = "Y1 Leaf 1/Scaled Actual (mm)"
    F_CY1 = "Y1 Leaf 80/Scaled Actual (mm)"
    I_Y1 = df_CSV.columns.get_loc(I_CY1)
    F_Y1 = df_CSV.columns.get_loc(F_CY1)
    MLC_I = df_CSV.columns[I_Y1:F_Y1 + 1]

    # Right MLC
    I_CY2 = "Y2 Leaf 1/Scaled Actual (mm)"
    F_CY2 = "Y2 Leaf 80/Scaled Actual (mm)"
    I_Y2 = df_CSV.columns.get_loc(I_CY2)
    F_Y2 = df_CSV.columns.get_loc(F_CY2)
    MLC_D = df_CSV.columns[I_Y2:F_Y2 + 1]

    # Group by CP (Radiation On)
    Y1 = F_RO.groupby(CSV_CP_C)[MLC_I].mean()
    CSV_Y1_MLC = (-1) * Y1.where(Y1 > -110, Y1 + 33.5)
    Y2 = F_RO.groupby(CSV_CP_C)[MLC_D].mean()
    CSV_Y2_MLC = Y2.where(Y2 < 110, Y2 - 33.5)

    # Group by CP (Move Only)
    Y1_MOVE = F_MO[MLC_I]
    Y2_MOVE = F_MO[MLC_D]
    TIME_MOVE = F_MO[Time_c]

    # Effective time 
    Time_E_min = F_DR.groupby(CSV_CP_C)[Time_c].min()
    Time_E_max = F_DR.groupby(CSV_CP_C)[Time_c].max()
    Time_Ef = (Time_E_max - Time_E_min).sum()  # min
    Total_Time = Time.max()
    P_Ef_Time = (Time_Ef * 100) / Total_Time

    # Dictionary with all data
    data_CSV = {
        "Control Point": CSV_CP,
        "Dose per CP": CSV_D_CP,
        "Total Dose [MU]": CSV_TD,
        "Total Time": round(Total_Time, 2),
        "Effective Time": round(Time_Ef, 2),
        "Percentage of Effective Time [%]": round(P_Ef_Time, 2),
        "Time in movement": TIME_MOVE,
        "Gantry Angle": CSV_GANTRY,
        "Jaw X1": CSV_JAWS_1,
        "Jaw X2": CSV_JAWS_2,
        "MLC Y1": CSV_Y1_MLC,
        "MLC Y2": CSV_Y2_MLC,
        "MLC Y1 MOV": Y1_MOVE,
        "MLC Y2 MOV": Y2_MOVE,
    }

    return df_CSV, data_CSV

In [ ]:
######## SELECT DCM FILE
R_DCM = select_file(
    title="Select DICOM RT PLAN file", 
    filetypes=(("DICOM files", "*.dcm"),)
    )

In [ ]:
######## EXTRACT DATA FROM A DICOM RT PLAN
df_DCM, data_DCM, info_Treat = dcm_data(R_DCM)

# Patient data
Patient = info_Treat["Patient Name"]
ID = info_Treat["Patient ID"]
Target = info_Treat["Target"]
Intento = info_Treat["Intent"]

# Geometry & Treatment data
DCM_CP_DF = data_DCM["Control Point"]
DCM_NH = data_DCM["Beam Number"]
DCM_TD = data_DCM["Total Dose"]
DCM_DH = data_DCM["Dose per Beam"]
DCM_DPC = data_DCM["Dose per CP"]
DCM_GANTRY = data_DCM["Gantry Angle"]
DCM_Y1_MLC = data_DCM["MLC Y1"]
DCM_Y2_MLC = data_DCM["MLC Y2"]
DCM_JAW_X1 = data_DCM["Jaw X1"]
DCM_JAW_X2 = data_DCM["Jaw X2"]

In [ ]:
######## SELECT TREATMENT LOG FILES (CSV)
root_csv_1 = select_csvs("Treatment")
df_CSV_concat_1 = concat_csvs(root_csv_1)

In [ ]:
######## SELECT ARCCHECK LOG FILES (CSV)
root_csv_2 = select_csvs("ArcCheck")
df_CSV_concat_2 = concat_csvs(root_csv_2)

In [ ]:
######## EXTRACT DATA FROM TREATMENT LOG FILES (CSV)
df_CSV_1, data_CSV_1 = csv_data(df_CSV_concat_1)

# Data:
CSV_CP_DF_1 = data_CSV_1["Control Point"]
CSV_TD_1 = data_CSV_1["Total Dose [MU]"]
CSV_D_CP_1 = data_CSV_1["Dose per CP"]
CSV_GANTRY_1 = data_CSV_1["Gantry Angle"]
CSV_Y1_MLC_1 = data_CSV_1["MLC Y1"]
CSV_Y2_MLC_1 = data_CSV_1["MLC Y2"]
Y1_MOVE_1 = data_CSV_1["MLC Y1 MOV"]
Y2_MOVE_1 = data_CSV_1["MLC Y2 MOV"]
CSV_JAW_X1_1 = data_CSV_1["Jaw X1"]
CSV_JAW_X2_1 = data_CSV_1["Jaw X2"]
CSV_TIME_TOTAL_1 = data_CSV_1["Total Time"]
CSV_TIME_EF_1 = data_CSV_1["Effective Time"]
CSV_P_TOTAL_1 = data_CSV_1["Percentage of Effective Time [%]"]
TIME_MOV_1 = data_CSV_1["Time in movement"]

In [ ]:
######## EXTRACT DATA FROM ARCCHECK LOG FILES (CSV)
df_CSV_2, data_CSV_2 = csv_data(df_CSV_concat_2)

# Data:
CSV_CP_DF_2 = data_CSV_2["Control Point"]
CSV_TD_2 = data_CSV_2["Total Dose [MU]"]
CSV_D_CP_2 = data_CSV_2["Dose per CP"]
CSV_GANTRY_2 = data_CSV_2["Gantry Angle"]
CSV_Y1_MLC_2 = data_CSV_2["MLC Y1"]
CSV_Y2_MLC_2 = data_CSV_2["MLC Y2"]
Y1_MOVE_2 = data_CSV_2["MLC Y1 MOV"]
Y2_MOVE_2 = data_CSV_2["MLC Y2 MOV"]
CSV_JAW_X1_2 = data_CSV_2["Jaw X1"]
CSV_JAW_X2_2 = data_CSV_2["Jaw X2"]
CSV_TIME_TOTAL_2 = data_CSV_2["Total Time"]
CSV_TIME_EF_2 = data_CSV_2["Effective Time"]
CSV_P_TOTAL_2 = data_CSV_2["Percentage of Effective Time [%]"]
TIME_MOV_2 = data_CSV_2["Time in movement"]

In [ ]:
######## INTERSECT CP (RADIATION ON)
def CPC(CP_DCM, MLC_CSV):
    C_CP = np.intersect1d(CP_DCM, MLC_CSV.index)
    return C_CP

In [ ]:
common_CP1 = CPC(DCM_CP_DF, CSV_Y1_MLC_1)
common_CP2 = CPC(DCM_CP_DF, CSV_Y1_MLC_2)
common_CP = np.intersect1d(common_CP1, common_CP2)

In [ ]:
######## FILTER MLC BY CP (RADIATION ON)

# DICOM
DCM_Y1 = DCM_Y1_MLC.iloc[common_CP - 1]
DCM_Y2 = DCM_Y2_MLC.iloc[common_CP - 1]

# CSV TREATMENT
CSV_Y1_1 = CSV_Y1_MLC_1.loc[common_CP]
CSV_Y2_1 = CSV_Y2_MLC_1.loc[common_CP]

# CSV ARCCHECK 
CSV_Y1_2 = CSV_Y1_MLC_2.loc[common_CP]
CSV_Y2_2 = CSV_Y2_MLC_2.loc[common_CP]

In [ ]:
######## FILTER JAWS BY CP (RADIATION ON)

# DCM
DCM_X1 = DCM_JAW_X1.iloc[common_CP - 1]
DCM_X2 = DCM_JAW_X2.iloc[common_CP - 1]

# CSV TREATMENT
CSV_X1_1 = CSV_JAW_X1_1.loc[common_CP]
CSV_X2_1 = CSV_JAW_X2_1.loc[common_CP]

# CSV ARCCHECK
CSV_X1_2 = CSV_JAW_X1_2.loc[common_CP]
CSV_X2_2 = CSV_JAW_X2_2.loc[common_CP]

In [ ]:
######## FILTER GANTRY ANGLE BY CP (RADIATION ON)

# DCM
DCM_GTY = DCM_GANTRY.iloc[common_CP - 1]

# CSV TREATMENT
CSV_GTY_1 = CSV_GANTRY_1.loc[common_CP]

# CSV ARCCHECK
CSV_GTY_2 = CSV_GANTRY_2.loc[common_CP]

In [ ]:
######### STATISTICS
def STAT(DIF, delta=1.0, alpha=0.05):
    
    # VECTOR 
    DIF = np.array(DIF).flatten()

    Vesp = 0
    t_stat, p_value = stats.ttest_1samp(DIF, Vesp)
    
    # STATS
    mean = np.mean(DIF)
    std = np.std(DIF, ddof=1)
    dsw = DescrStatsW(DIF)
    
    # CONFIDENCE INTERVAL
    ci_low, ci_high = dsw.tconfint_mean(alpha=alpha)
    
    # TOST
    low, high = -delta, delta
    p_lower = dsw.ttest_mean(low, alternative='larger')[1]
    p_upper = dsw.ttest_mean(high, alternative='smaller')[1] 
    p_tost = max(p_lower, p_upper)
    
    return {
        "Mean": np.round(mean, 5),
        "SD": np.round(std, 5),
        "T-Student": np.round(t_stat, 5),
        "P-value": np.round(p_value, 5),
        "ci": np.round((ci_low, ci_high), 5),
        "p_tost": np.round(p_tost, 5)
    }

In [ ]:
######## DIFERENCE OF MU
def D_MU(D_DCM, D_CSV):
    root = tk.Tk()
    root.withdraw()
    
    Dif_D = D_DCM.max() - D_CSV
    E_PER = abs(round(Dif_D*100/D_DCM.max(), 2))
    print(
        f"Monitor units DICOM: {round(D_DCM.max(),2)} MU"
        )
    print(
        f"Monitor units log file: {round(D_CSV,2)} MU"
        )
    print(
        f"Difference of monitor units: {round(Dif_D,2)} MU"
        )
    print(
        f"Percent difference: {E_PER} %"
        )
    return Dif_D

In [ ]:
###### DIF. MU: DICOM vs Tratamiento
E_D_1 = D_MU(DCM_TD, CSV_TD_1)

In [ ]:
###### DIF. MU: DICOM vs PSQA
E_D_2 = D_MU(DCM_TD, CSV_TD_2)

In [ ]:
####### DIF. UM TREATMENT vs PSQA
def MU_TA(MU_T, MU_A):
    DIF = MU_T - MU_A
    print(f"Difference in MU: treatment vs PSQA: {DIF} MU")

In [ ]:
MU_TA(E_D_1, E_D_2)

In [ ]:
######## DIF. MLC (DCM vs CSV)
def DMLC(MLC_DCM, MLC_CSV):
    DIF = MLC_DCM.values - MLC_CSV.values
    return DIF

In [ ]:
######## DIF. MLC (BANK Y1 & Y2)

# TREATMENT
Dif_Y1_1 = DMLC(DCM_Y1, CSV_Y1_1)   #LEFT
Dif_Y2_1 = DMLC(DCM_Y2, CSV_Y2_1)   #RIGHT

# ARCCHECK
Dif_Y1_2 = DMLC(DCM_Y1, CSV_Y1_2)   #LEFT
Dif_Y2_2 = DMLC(DCM_Y2, CSV_Y2_2)   #RIGHT

In [ ]:
######## ERROR MLC
def D_MLC(D_MLC1, D_MLC2, Y):
    N_L1 = D_MLC1.shape[1] + 1
    N_L2 = D_MLC2.shape[1] + 1
    N1 = np.arange(1, N_L1)
    N2 = np.arange(1, N_L2)
    E_P_MLC_1 = D_MLC1.mean(axis=0)
    E_P_MLC_2 = D_MLC2.mean(axis=0)
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.scatter(N1, E_P_MLC_1, marker='o', label='Treatment', color='blue', alpha=0.7)
    ax.scatter(N2, E_P_MLC_2, marker='^' , label='ArcCheck', color='orange', alpha=0.7)
    ax.set_xlabel("Leaf number")
    ax.set_ylabel("Average difference [mm]")
    ax.set_title(f"Average difference per leaf of the MLC (bank {Y})")
    ax.axhline(1, color='g', linestyle='--')
    ax.axhline(0, color='black', linestyle='--')
    ax.axhline(-1, color='g', linestyle='--')
    ax.set_xticks(N1)
    ax.tick_params(rotation=90)
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
###### DIF. MLC Y1: TREATMENT vs ARCCHECK
E_Y1 = D_MLC(Dif_Y1_1, Dif_Y1_2, "Y1")

In [ ]:
###### ERROR MLC Y1: TREATMENT vs ARCCHECK
D_1_1 = Dif_Y1_1.mean(axis=0)
D_1_2 = Dif_Y1_2.mean(axis=0)
DIF_Y1D = D_1_1 - D_1_2
STAT(DIF_Y1D, delta=1.0)

In [ ]:
###### DIF MLC Y2: TREATMENT vs ARCCHECK
E_Y2 = D_MLC(Dif_Y2_1, Dif_Y2_2, "Y2")

In [ ]:
###### ERROR MLC Y2: TREATMENT vs ARCCHECK
D_2_1 = Dif_Y2_1.mean(axis=0)
D_2_2 = Dif_Y2_2.mean(axis=0)
DIF_Y2D = D_2_1 - D_2_2
STAT(DIF_Y2D, delta=1.0)

In [ ]:
######## VIOLINPLOT WITHOUT OUTLIERS
def VIO_P(D_MLC, Y):
    N_L = D_MLC.shape[1]
    # DataFrame
    df_dif_MLC = pd.DataFrame(D_MLC, columns=[i+1 for i in range(N_L)])
    df_long = df_dif_MLC.melt(var_name="MLC", value_name="Error [mm]")

    # IQR (InterQuartile Range)  
    Q1 = df_long["Error [mm]"].quantile(0.05)    #QUARTILE 1
    Q3 = df_long["Error [mm]"].quantile(0.95)    #QUARTILE 2
    IQR = Q3 - Q1                                #InterQuartile Range
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Filter data
    df_long = df_long[(df_long["Error [mm]"] >= lower) & 
                    (df_long["Error [mm]"] <= upper)]

    # VIOLINPLOTS
    plt.figure(figsize=(15, 6))
    sns.violinplot(
        x="MLC", y="Error [mm]", 
        data=df_long, 
        inner="quartile", density_norm="width", cut=0
    )
    plt.xticks(rotation=90)
    plt.title(f"Distribution of differences per leaf of bank {Y} of the MLC")
    plt.xlabel("Leaf number")
    plt.ylabel("Position difference [mm]")
    plt.show()

In [ ]:
######## VIOLINPLOT BANK Y1
V_PLOT_Y1_1 = VIO_P(Dif_Y1_1, "Y1-Treatment")
V_PLOT_Y1_2 = VIO_P(Dif_Y1_2, "Y1-ArcCheck")

In [ ]:
######## VIOLINPLOT BANK Y2
V_PLOT_Y2_1 = VIO_P(Dif_Y2_1, "Y2-Treatment")
V_PLOT_Y2_2 = VIO_P(Dif_Y2_2, "Y2-ArcCheck")

In [ ]:
####### AREA BY MLC
def Area(Y1_MLC, Y2_MLC):
    DIST = Y1_MLC.values - Y2_MLC.values
    AREA = abs((DIST*5).sum(axis=1)).tolist()
    return AREA  

In [ ]:
######## AREAS
DCM_AREA = Area(DCM_Y1, DCM_Y2)
CSV_AREA_1 = Area(CSV_Y1_1, CSV_Y2_1)
CSV_AREA_2 = Area(CSV_Y1_2, CSV_Y2_2)

In [ ]:
####### DIF. AREAS
def A_E(C_CP, AREA_DCM, AREA_CSV1, AREA_CSV2):
    DIF_A1 = np.array(AREA_DCM) - np.array(AREA_CSV1)
    DIF_A2 = np.array(AREA_DCM) - np.array(AREA_CSV2)
    DIF_A = round((DIF_A1 - DIF_A2).mean(),5)
    # SCATTER
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.scatter(C_CP, DIF_A1, marker='o', label='Treatment', color='blue', alpha=0.7)
    ax.scatter(C_CP, DIF_A2, marker='^', label='ArcCheck', color='orange', alpha=0.7)
    ax.set_xlabel("Control point")
    ax.set_ylabel("Difference of areas [mm^2]")
    ax.set_title("Areas difference per control point")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Average difference: {DIF_A} mm^2")

In [ ]:
ERROR_A = A_E(common_CP, DCM_AREA, CSV_AREA_1, CSV_AREA_2)

In [ ]:
######## DIF. JAWS
def E_J(C_CP, JAW_DCM, JAW_CSV1, JAW_CSV2, X):
    DIF_J1 = JAW_DCM.values - JAW_CSV1.values
    DIF_J2 = JAW_DCM.values - JAW_CSV2.values
    DIF_J = round((DIF_J1 - DIF_J2).mean(),5)
    # SCATTER
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.scatter(C_CP, DIF_J1, marker='o', label='Treatment', color='blue', alpha=0.7)
    ax.scatter(C_CP, DIF_J2, marker='^', label='ArcCheck', color='orange', alpha=0.7)
    ax.set_xlabel("Control point")
    ax.set_ylabel("Average difference [mm]")
    ax.set_title(f"Jaw position difference-{X} per control point")
    ax.axhline(1, color='g', linestyle='--')
    ax.axhline(0, color='black', linestyle='--')
    ax.axhline(-1, color='g', linestyle='--')
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
######## JAW X1
E_X1 = E_J(common_CP, DCM_X1, CSV_X1_1, CSV_X1_2, "X1")
DIF_J11 = DCM_X1.values - CSV_X1_1.values #TREATMENT
DIF_J12 = DCM_X1.values - CSV_X1_2.values #PSQA
DIF_J1 = DIF_J11 - DIF_J12
STAT(DIF_J1, delta=1.0)

In [ ]:
######## JAW X2
E_X2 = E_J(common_CP, DCM_X2, CSV_X2_1, CSV_X2_2, "X2")
DIF_J21 = DCM_X2.values - CSV_X2_1.values
DIF_J22 = DCM_X2.values - CSV_X2_2.values
DIF_J2 = DIF_J21 - DIF_J22
STAT(DIF_J2, delta=1.0)

In [ ]:
######## DIF.GANTRY
def GE(C_CP, DCM_G, CSV_G1, CSV_G2):
    DIF_G1 = DCM_G.values - CSV_G1.values
    DIF_G2 = DCM_G.values - CSV_G2.values
    DIF_G = round((DIF_G1 - DIF_G2).mean(),10)
    # SCATTER
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.scatter(C_CP, DIF_G1, marker='o', label='Treatment', color='blue', alpha=0.7)
    ax.scatter(C_CP, DIF_G2, marker='^', label='ArcCheck', color='orange', alpha=0.7)
    ax.set_xlabel("Control point")
    ax.set_ylabel("Average difference [grades]")
    ax.set_title("Angular position difference of the gantry per control point")
    ax.axhline(0, color='black', linestyle='--')
    ax.axhline(0.5, color='g', linestyle='--')
    ax.axhline(-0.5, color='g', linestyle='--')
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
######## GANTRY
E_G = GE(common_CP, DCM_GTY, CSV_GTY_1, CSV_GTY_2)
DIF_G1 = DCM_GTY.values - CSV_GTY_1.values
DIF_G2 = DCM_GTY.values - CSV_GTY_2.values
DIF_G =  DIF_G1 - DIF_G2
STAT(DIF_G, delta=0.5)

In [ ]:
######## MLC SPEED
def spd_mlc(Y, Y_M_1, T_M_1, Y_M_2, T_M_2):
    # Y1
    Dif_pos_1 = abs(Y_M_1.diff().fillna(0))
    Dif_time_1 = T_M_1.diff().fillna(0.04)
    SPD_1 = Dif_pos_1.values/Dif_time_1.values[:, np.newaxis]
    SPD_Mean_1 = SPD_1.mean(axis=0)
    SPD_M_1 = np.round(SPD_Mean_1.mean(),5)
    SPD_MAX_1 = np.round(SPD_Mean_1.max(),5)
    N_L_1 = Y_M_1.shape[1] + 1
    N_1 = np.arange(1, N_L_1)
    stat_1 = STAT(SPD_Mean_1)
    # Y2
    Dif_pos_2 = abs(Y_M_2.diff().fillna(0))
    Dif_time_2 = T_M_2.diff().fillna(0.04)
    SPD_2 = Dif_pos_2.values/Dif_time_2.values[:, np.newaxis]
    SPD_Mean_2 = SPD_2.mean(axis=0)
    SPD_M_2 = np.round(SPD_Mean_2.mean(),5)
    SPD_MAX_2 = np.round(SPD_Mean_2.max(),5)
    N_L_2 = Y_M_2.shape[1] + 1
    N_2 = np.arange(1, N_L_2)
    stat_2 = STAT(SPD_Mean_2)
    # SCATTER
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.scatter(N_1, SPD_Mean_1, marker='o', label='Treatment', color='blue', alpha=0.7)
    ax.scatter(N_2, SPD_Mean_2, marker='^', label='ArcCheck', color='orange', alpha=0.7)
    ax.set_xlabel("Leaf number")
    ax.set_ylabel("Average speed [mm/s]")
    ax.set_title(f"Average speed per leaf of bank {Y} of the MLC")
    ax.set_xticks(N_1)
    ax.tick_params(rotation=90)
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Average speed of bank {Y} (Treatment): {SPD_M_1} mm/s")
    print(f"Max. speed of bank {Y} (Treatment): {SPD_MAX_1} mm/s")
    print(f"SD (Treatment): {stat_1['SD']:.5f} mm/s")
    print(f"P-value (Treatment): {stat_1['P-value']:.5e}")
    print(f"Average speed of bank {Y} (ArcCheck): {SPD_M_2} mm/s")
    print(f"Max. speed of bank {Y} (ArcCheck): {SPD_MAX_2} mm/s")
    print(f"SD (ArcCheck): {stat_2['SD']:.5f} mm/s")
    print(f"P-value (ArcCheck): {stat_2['P-value']:.5e}")


In [ ]:
######## SPEED BANK Y1
SPD_Y1 = spd_mlc("Y1", Y1_MOVE_1, TIME_MOV_1, Y1_MOVE_2, TIME_MOV_2)

In [ ]:
######## SPEED BANK Y2
SPD_Y2 = spd_mlc("Y2", Y2_MOVE_1, TIME_MOV_1, Y2_MOVE_2, TIME_MOV_2)

In [ ]:
######## MODULATION COMPLEXITY SCORE
def Dist(I_MLC, D_MLC):
    DIST = D_MLC.values - I_MLC.values
    return DIST

# Aperture Area Variability (AAV)
def AAV_i(DIF_MLC):
    NMLC = DIF_MLC.shape[1]
    S_Ap = DIF_MLC.sum(axis=1)
    Max_Ap = np.max(DIF_MLC, axis=1)*NMLC
    Max_Ap[Max_Ap == 0] = np.nan
    AAV_list = np.array(S_Ap/Max_Ap)
    return AAV_list

# Leaf Sequence Variability (LSV)
def LSV_i(POS_MLC):
    NF = POS_MLC.shape[1]
    Dif_F = np.abs(np.diff(POS_MLC.values, axis=1))
    P_max = np.max(np.abs(POS_MLC))
    if P_max == 0:
        return np.full(POS_MLC.shape[0], np.nan)
    LSV_list = np.array(np.sum(P_max - Dif_F, axis=1)/((NF - 1)* P_max))
    return LSV_list

# Modulation Complexity Score (MCS)
def MCS(AAV, LSV, MU):
    MCS_s = 0
    mu = np.array(MU)
    MU_T = np.sum(MU)
    if MU_T == 0:
        return np.nan
    for i in range(len(MU) - 1):
        aav_avg = (AAV[i] + AAV[i+1])/2
        lsv_avg = (LSV[i] + LSV[i+1])/2
        mu_r = (mu[i])/MU_T
        MCS_s += aav_avg*lsv_avg*mu_r

    return {
        "MCS_val": MCS_s
    }

In [ ]:
######## METRIC AAV
# TREATMENT
CSV_DIST_1 = Dist(CSV_Y1_MLC_1, CSV_Y2_MLC_1)
AAV_CSV_1 = AAV_i(CSV_DIST_1)
# PSQA
CSV_DIST_2 = Dist(CSV_Y1_MLC_2, CSV_Y2_MLC_2)
AAV_CSV_2 = AAV_i(CSV_DIST_2)

In [ ]:
######## MCS BANK Y1 (TREATMENT)
LSV_CSV_Y1_1 = LSV_i(CSV_Y1_MLC_1)
MCS_CSV_Y1_1 = MCS(AAV_CSV_1, LSV_CSV_Y1_1, CSV_D_CP_1)
######## MCS BANK Y2 (TREATMENT)
LSV_CSV_Y2_1 = LSV_i(CSV_Y2_MLC_1)
MCS_CSV_Y2_1 = MCS(AAV_CSV_1, LSV_CSV_Y2_1, CSV_D_CP_1)

In [ ]:
######## MCS BANK Y1 (PSQA)
LSV_CSV_Y1_2 = LSV_i(CSV_Y1_MLC_2)
MCS_CSV_Y1_2 = MCS(AAV_CSV_2, LSV_CSV_Y1_2, CSV_D_CP_2)
######## MCS BANK Y2 (PSQA)
LSV_CSV_Y2_2 = LSV_i(CSV_Y2_MLC_2)
MCS_CSV_Y2_2 = MCS(AAV_CSV_2, LSV_CSV_Y2_2, CSV_D_CP_2)

In [ ]:
######## MCS (MEAN VALUE)
def MCS_M(MCS_1, MCS_2, TvsA):
    MCS_MEAN = round((MCS_1["MCS_val"] + MCS_2["MCS_val"])/2, 5)
    print(f"MCS {TvsA}: {MCS_MEAN}")

In [ ]:
######## MCS (TREATMENT)
MCS_Mean_TRT = MCS_M(MCS_CSV_Y1_1, MCS_CSV_Y2_1, "treatment")
######## MCS (PSQA)
MCS_Mean_PSQA = MCS_M(MCS_CSV_Y1_2, MCS_CSV_Y2_2, "PSQA")